In [7]:
import torch
import yaml
import pickle

def config_loader(config_file_location):

    f = open(config_file_location)
    #config = yaml.load(f, Loader=yaml.FullLoader)
    config = yaml.safe_load(f)
    f.close()
    config['device'] = torch.device(config['device'])
    
    if config['model_name'] == 'GPNN_icra':
        num_frames = int(config['num_frames'])
        config['edge_feature_size'] = 7 * num_frames

    return config



def convert_tensor_values_to_float(input_dict):
    """
    Converts all values in the dictionary that are tensors to float dtype,
    except for tensors of dtype torch.long.
    """
    # Initialize an empty dictionary to store the results
    output_dict = {}
    
    for key, value in input_dict.items():
        # Check if the value is a tensor and not of type torch.long
        if isinstance(value, torch.Tensor) and value.dtype != torch.long:
            # Convert the tensor value to float and update in the new dictionary
            output_dict[key] = value.float()
        else:
            # If the value is not a tensor or is of type torch.long, 
            # directly update in the new dictionary
            output_dict[key] = value
    
    return output_dict

def process_data_for_fpass(data_item, config):
    
    all_keys = data_item.keys()
    
    for k in all_keys:
        if isinstance(data_item[k], int):
            data_item[k] = torch.tensor([data_item[k]])
            continue
        if isinstance(data_item[k], torch.Tensor):
            data_item[k] = data_item[k].unsqueeze(0)
            continue
    
    data_item['num_relation'] = data_item['num_relation'].to(config['device'])
    data_item['num_obj'] = data_item['num_obj'].to(config['device'])

    data_item['object_pairs'] = data_item['object_pairs'].type(torch.long)
    
    # Padded features
    obj_features = []
    
    for f in config['features_list']:
        
        if f in config['custom_filter_dict'].keys():
            
            # increased by 1 to take care of the batching dimension
            frame_dim = config['custom_filter_dict'][f]['frame_dim'] + 1
            
            frame_index = config['custom_filter_dict'][f]['frame_index']
            frame_index = torch.tensor([frame_index])
            
            temp_feat = data_item[f].index_select( 
                                                    dim = frame_dim,
                                                    index = frame_index
                                                ).squeeze().to(config['device']).unsqueeze(0)
            
            obj_features.append(temp_feat)
            print(f)
        
        else:
            temp_feat = data_item[f].to(config['device'])
            obj_features.append(temp_feat)
            print(f)
    # return obj_features
    for k in obj_features:
        print("DEBUG", k.shape)
    obj_features = torch.cat(obj_features, 2)

    data_item['concatenated_node_features'] = obj_features.to(config['device'])

    interaction_centric_features = []
    
    for f in config['relative_features_list']:
        temp_feat = data_item[f].flatten(3).to(config['device'])
        interaction_centric_features.append(temp_feat)

    data_item['interaction_feature'] = torch.cat(interaction_centric_features, 3)
    
    data_item = convert_tensor_values_to_float(data_item)

    return data_item

import pickle as pkl
import torch


config = config_loader('/workspace/work/misc/O2ONet/sota_experiments/gnn_revise_resubmit_v3/configs/gpnn.yaml')
inf_features = pkl.load(open('/workspace/work/ral_revise_and_resubmit/ral_revise_resubmit/revised_feat_extraction/sample_inference_feature.pkl', 'rb'))

# Process the data for forward pass (assuming `process_data_for_fpass` takes the batch and config)
d_item = process_data_for_fpass(inf_features, config)


object_i3d_feature
bbox_CLIP
geometric_feature
object_semantic_embeddings
object_centric_shape_feats
DEBUG torch.Size([1, 12, 2048])
DEBUG torch.Size([1, 12, 768])
DEBUG torch.Size([1, 12, 5])
DEBUG torch.Size([1, 12, 300])
DEBUG torch.Size([1, 12, 16])


NameError: name 'convert_tensor_values_to_float' is not defined

In [10]:
inf_features.keys()

dict_keys(['legend', 'metadata', 'num_obj', 'bboxes', 'object_pairs', 'num_relation', 'geometric_feature', '2d_cnn_feature_map', 'object_2d_cnn_feature', 'iou', 'distance', 'relative_spatial_feature', 'i3d_feature_map', 'object_i3d_feature', 'motion_feature', 'bbox_CLIP', 'sam_masks', 'object_semantic_embeddings', 'interaction_bbox_CLIP', 'object_centric_shape_feats', 'interaction_centric_shape_feats'])

In [19]:

for i in range(len(d_item)):
    print(d_item[i].shape)

torch.Size([12, 2048])
torch.Size([1, 12, 768])
torch.Size([12, 5])
torch.Size([1, 12, 300])
torch.Size([12, 16])


In [3]:
t = torch.randn(1,2,3,1,4,1)

In [6]:
t.squeeze().shape

torch.Size([2, 3, 4])